# SetWise - Exercise Classifier v5

**Task**: exercise classification only.

**Design**
- All inputs are resampled to `T_NORM = 512` timesteps at `50 Hz`
- Single pooled `train / val / test` split across `Whales1and2 + mm-fit + recofit`
- Single classification objective (`CrossEntropyLoss`) with no rep-count head

**Included datasets**
- `Whales1and2` (wrist, 100 Hz -> 50 Hz)
- `mm-fit` (wrist, 50 Hz)
- `recofit` (forearm, 50 Hz)

**Dropped**
- `INSIGHT-LME`: not used in v5
- Rep counting / multi-task training: removed entirely

**Notes**
- This notebook keeps the two unknown Whales labels (`SBLP`, `DWC`) excluded until their exercise mapping is confirmed.
- The split is segment-level, not participant-holdout, so test accuracy may be optimistic.

In [ ]:
# Download datasets from Google Drive to JupyterHub
# Fill in the IDs from your Drive share links only if the local folders/files are missing.
#
# Folder link ID: the string after /folders/
# File link ID:   the string between /d/ and /view

import os
import subprocess
import sys

WHALES_FOLDER_ID = 'PASTE_WHALES1AND2_FOLDER_ID_HERE'
RECOFIT_FILE_ID  = 'PASTE_RECOFIT_MAT_FILE_ID_HERE'
MMFIT_FOLDER_ID  = 'PASTE_MMFIT_FOLDER_ID_HERE'


def resolve_data_root():
    candidates = [
        os.getcwd(),
        '/home/jovyan/SetWise',
        os.path.expanduser('~/0SetWise'),
        os.path.expanduser('~/git/0SetWise'),
    ]
    for root in candidates:
        whales_dir = os.path.join(root, 'Whales1and2')
        recofit_mat = os.path.join(root, 'recofit', 'exercise_data.50.0000_singleonly.mat')
        mmfit_dir = os.path.join(root, 'mm-fit')
        if os.path.isdir(whales_dir) or os.path.exists(recofit_mat) or os.path.isdir(mmfit_dir):
            return root
    return candidates[0]


DATA_ROOT = resolve_data_root()
whales_dir = os.path.join(DATA_ROOT, 'Whales1and2')
recofit_dir = os.path.join(DATA_ROOT, 'recofit')
recofit_mat = os.path.join(recofit_dir, 'exercise_data.50.0000_singleonly.mat')
mmfit_dir = os.path.join(DATA_ROOT, 'mm-fit')

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(recofit_dir, exist_ok=True)

need_whales = not os.path.isdir(whales_dir) or not os.listdir(whales_dir)
need_recofit = not os.path.exists(recofit_mat)
need_mmfit = not os.path.isdir(mmfit_dir) or not os.listdir(mmfit_dir)


def _require_id(label, value):
    if value.startswith('PASTE_'):
        raise ValueError(f'{label} is missing. Paste the Drive ID into this cell before running it.')


if need_whales or need_recofit or need_mmfit:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown

    if need_whales:
        _require_id('WHALES_FOLDER_ID', WHALES_FOLDER_ID)
        print('Downloading Whales1and2 folder...')
        gdown.download_folder(id=WHALES_FOLDER_ID, output=DATA_ROOT, quiet=False, resume=True)
    else:
        print(f'Whales1and2 already present ({len(os.listdir(whales_dir))} files)')

    if need_recofit:
        _require_id('RECOFIT_FILE_ID', RECOFIT_FILE_ID)
        print('Downloading recofit .mat file (~1.5 GB)...')
        gdown.download(id=RECOFIT_FILE_ID, output=recofit_mat, quiet=False, resume=True)
    else:
        print(f'recofit already present ({os.path.getsize(recofit_mat) // 1_000_000} MB)')

    if need_mmfit:
        _require_id('MMFIT_FOLDER_ID', MMFIT_FOLDER_ID)
        print('Downloading mm-fit folder...')
        gdown.download_folder(id=MMFIT_FOLDER_ID, output=DATA_ROOT, quiet=False, resume=True)
    else:
        print(f'mm-fit already present ({len(os.listdir(mmfit_dir))} items)')
else:
    print('All datasets already present locally.')

print(f'\nDATA_ROOT: {DATA_ROOT}')
print(f'  Whales1and2 : {os.path.isdir(whales_dir)}')
print(f'  recofit     : {os.path.exists(recofit_mat)}')
print(f'  mm-fit      : {os.path.isdir(mmfit_dir)}')

In [ ]:
import gc
import os
import pathlib
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import scipy.io
from scipy.signal import resample as scipy_resample
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset


def resolve_data_root():
    candidates = [
        os.getcwd(),
        '/home/jovyan/SetWise',
        os.path.expanduser('~/0SetWise'),
        os.path.expanduser('~/git/0SetWise'),
    ]
    for root in candidates:
        whales_dir = os.path.join(root, 'Whales1and2')
        recofit_mat = os.path.join(root, 'recofit', 'exercise_data.50.0000_singleonly.mat')
        mmfit_dir = os.path.join(root, 'mm-fit')
        if os.path.isdir(whales_dir) or os.path.exists(recofit_mat) or os.path.isdir(mmfit_dir):
            return root
    return candidates[0]


try:
    DATA_ROOT
except NameError:
    DATA_ROOT = resolve_data_root()

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/0SetWise (1)'
except Exception:
    pass

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'  Whales1and2 exists : {os.path.isdir(os.path.join(DATA_ROOT, "Whales1and2"))}')
print(f'  recofit exists     : {os.path.exists(os.path.join(DATA_ROOT, "recofit", "exercise_data.50.0000_singleonly.mat"))}')
print(f'  mm-fit exists      : {os.path.isdir(os.path.join(DATA_ROOT, "mm-fit"))}')
print('Device:', device)

## Configuration & Exercise Taxonomy

In [ ]:
BASE = pathlib.Path(DATA_ROOT)

TARGET_HZ = 50
T_NORM = 512
N_CHANNELS = 6
MIN_CLASS_COUNT = 40
BATCH = 16          # Lower this to 8 if your GPU is tight on memory.
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 12

EXERCISE_MAP = {
    # mm-fit (wrist)
    'squats': 'squats',
    'lunges': 'lunges',
    'bicep_curls': 'bicep_curls',
    'situps': 'situps',
    'pushups': 'pushups',
    'tricep_extensions': 'tricep_extensions',
    'dumbbell_rows': 'rows',
    'jumping_jacks': 'jumping_jacks',
    'dumbbell_shoulder_press': 'shoulder_press',
    'lateral_shoulder_raises': 'lateral_raises',

    # Whales1and2 (wrist)
    'APULL': 'pullups',
    'CGCR': 'rows',
    'CGOCTE': 'tricep_extensions',
    'DLR': 'lateral_raises',
    'DSP': 'shoulder_press',
    'IDBC': 'bicep_curls',
    'AIDBC': 'bicep_curls',
    'MGTBR': 'rows',
    'MIBP': 'bench_press',
    'MSP': 'shoulder_press',
    'MTE': 'tricep_extensions',
    'NGCR': 'rows',
    'PREC': 'bicep_curls',
    'SACLR': 'lateral_raises',
    'SAOCTE': 'tricep_extensions',
    'SAODTE': 'tricep_extensions',
    '30BP': 'bench_press',
    '30DBP': 'bench_press',
    '45DBP': 'bench_press',

}

# Classes excluded due to prediction bias or signal ambiguity.
# jumping_jacks: 13% precision — predicted everywhere, signal too ambiguous across datasets.
# pullups: 15% precision — small sample count, dominates false positives.
EXCLUDE_CLASSES = {'jumping_jacks', 'pullups'}

# Remove excluded classes from the map
EXERCISE_MAP = {k: v for k, v in EXERCISE_MAP.items() if v not in EXCLUDE_CLASSES}

print(f'BASE: {BASE}')
print(f'T_NORM={T_NORM}  TARGET_HZ={TARGET_HZ}  BATCH={BATCH}')

## Dataset Loaders

In [ ]:
def _downsample_to_target(arr, src_hz):
    if src_hz == TARGET_HZ:
        return arr
    step = src_hz // TARGET_HZ
    return arr[::step]


def load_whales(base=BASE):
    wdir = pathlib.Path(base) / 'Whales1and2'
    if not wdir.is_dir():
        print('Whales1and2  :     0 segments (not found)')
        return []

    acc_cols = [
        'wristMotion_accelerationX',
        'wristMotion_accelerationY',
        'wristMotion_accelerationZ',
    ]
    gyr_cols = [
        'wristMotion_rotationRateX',
        'wristMotion_rotationRateY',
        'wristMotion_rotationRateZ',
    ]

    segments = []
    dropped_labels = Counter()

    for csv_path in sorted(wdir.glob('*.csv')):
        if csv_path.name.startswith('._'): continue  # skip macOS resource forks
        df = pd.read_csv(csv_path, encoding='latin-1').dropna(subset=acc_cols + gyr_cols)
        if len(df) < 50:
            continue

        raw_label = str(df['activity'].iloc[0])
        if raw_label not in EXERCISE_MAP:
            dropped_labels[raw_label] += 1
            continue

        imu = df[acc_cols + gyr_cols].to_numpy(dtype=np.float32)
        imu = _downsample_to_target(imu, src_hz=100)
        segments.append({
            'imu': imu,
            'exercise': EXERCISE_MAP[raw_label],
            'source': 'whales',
            'record_id': csv_path.stem,
        })

    print(f'Whales1and2  : {len(segments):>5} segments')
    if dropped_labels:
        print('  unmapped labels:', dict(dropped_labels))
    return segments


def load_mmfit(base=BASE):
    mmdir = pathlib.Path(base) / 'mm-fit'
    if not mmdir.is_dir():
        print('mm-fit       :     0 segments (not found)')
        return []

    segments = []
    for worker in range(21):
        pdir = mmdir / f'w{worker:02d}'
        label_path = pdir / f'w{worker:02d}_labels.csv'
        acc_path = pdir / f'w{worker:02d}_sw_l_acc.npy'
        gyr_path = pdir / f'w{worker:02d}_sw_l_gyr.npy'
        if not all(p.exists() for p in [label_path, acc_path, gyr_path]):
            continue

        labels = pd.read_csv(
            label_path,
            header=None,
            names=['start_frame', 'end_frame', 'reps', 'exercise_name'],
        )
        acc = np.load(acc_path)
        gyr = np.load(gyr_path)

        for seg_idx, row in labels.iterrows():
            raw_label = str(row['exercise_name'])
            if raw_label not in EXERCISE_MAP:
                continue

            start = int(row['start_frame'])
            end = int(row['end_frame'])
            imu = np.concatenate([acc[start:end, 2:5], gyr[start:end, 2:5]], axis=1).astype(np.float32)
            if len(imu) < 20:
                continue

            segments.append({
                'imu': imu,
                'exercise': EXERCISE_MAP[raw_label],
                'source': 'mmfit',
                'record_id': f'w{worker:02d}_{seg_idx:04d}',
            })

    print(f'mm-fit       : {len(segments):>5} segments')
    return segments


def load_recofit(base=BASE):
    recofit_path = pathlib.Path(base) / 'recofit' / 'exercise_data.50.0000_singleonly.mat'
    if not recofit_path.exists():
        print('recofit      :     0 segments (not found)')
        return []

    print('Loading recofit (~3 min)...')
    mat = scipy.io.loadmat(recofit_path, struct_as_record=False, squeeze_me=True)
    activities = list(mat['exerciseConstants'].activities)
    subject_data = mat['subject_data']

    segments = []
    for subject_idx in range(subject_data.shape[0]):
        for activity_idx in range(subject_data.shape[1]):
            cell = subject_data[subject_idx, activity_idx]
            if isinstance(cell, (float, np.floating)) and cell == 0:
                continue

            raw_label = activities[activity_idx]
            if raw_label not in EXERCISE_MAP:
                continue

            records = [cell] if not hasattr(cell, '__len__') else cell
            for rec_idx, record in enumerate(records):
                try:
                    imu = np.concatenate([
                        record.data.accelDataMatrix[:, 1:4],
                        record.data.gyroDataMatrix[:, 1:4],
                    ], axis=1).astype(np.float32)
                except Exception:
                    continue

                if len(imu) < 20:
                    continue

                segments.append({
                    'imu': imu,
                    'exercise': EXERCISE_MAP[raw_label],
                    'source': 'recofit',
                    'record_id': f's{subject_idx:02d}_a{activity_idx:02d}_r{rec_idx:04d}',
                })

    del mat, subject_data
    gc.collect()
    print(f'recofit      : {len(segments):>5} segments')
    return segments

## Load All Datasets

In [ ]:
all_segments = load_whales() + load_mmfit()
if not all_segments:
    raise RuntimeError('No segments were loaded. Check DATA_ROOT and dataset files before continuing.')

print(f'\nTotal segments: {len(all_segments)}')

## Exercise Frequency Analysis & Filtering

In [ ]:
summary = pd.DataFrame({
    'exercise': [s['exercise'] for s in all_segments],
    'source': [s['source'] for s in all_segments],
    'n_samples': [len(s['imu']) for s in all_segments],
})

print('Segments by source:')
print(summary['source'].value_counts().sort_index().to_string())
print('\nMedian raw length by source:')
print(summary.groupby('source')['n_samples'].median().sort_index().to_string())

exercise_source = pd.crosstab(summary['exercise'], summary['source'])
exercise_source['total'] = exercise_source.sum(axis=1)
exercise_source = exercise_source.sort_values('total', ascending=False)
print('\nExercise distribution before filtering:')
display(exercise_source)

keep_exercises = set(exercise_source.index[exercise_source['total'] >= MIN_CLASS_COUNT])
all_segments = [s for s in all_segments if s['exercise'] in keep_exercises]
summary = summary[summary['exercise'].isin(keep_exercises)].reset_index(drop=True)

print(f'Kept {len(keep_exercises)} classes and {len(all_segments)} segments after filtering.')
print('Classes:', sorted(keep_exercises))

## Preprocessing & Split

The scaler is fit on the raw training timesteps only, then each segment is resampled to `T_NORM` and encoded for classification.

In [ ]:
label_encoder = LabelEncoder().fit(sorted(keep_exercises))
label_names = label_encoder.classes_

labels_all = label_encoder.transform([s['exercise'] for s in all_segments]).astype(np.int64)
sources_all = np.array([s['source'] for s in all_segments], dtype=object)
record_ids_all = np.array([s['record_id'] for s in all_segments], dtype=object)
indices = np.arange(len(all_segments))

idx_train_val, idx_test = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    stratify=labels_all,
)
idx_train, idx_val = train_test_split(
    idx_train_val,
    test_size=0.15 / 0.85,
    random_state=SEED,
    stratify=labels_all[idx_train_val],
)

print(f'Split sizes -> train: {len(idx_train)}, val: {len(idx_val)}, test: {len(idx_test)}')



def preprocess_segments(idxs):
    X = []
    for i in idxs:
        imu = all_segments[i]['imu'].copy().astype(np.float32)
        # Per-segment z-score normalization to handle mmfit vs whales scaling differences
        mean = imu.mean(axis=0, keepdims=True)
        std = imu.std(axis=0, keepdims=True)
        std[std < 1e-6] = 1e-6
        imu = (imu - mean) / std
        imu = scipy_resample(imu, T_NORM).astype(np.float32)
        X.append(imu)
    return np.stack(X)


X_train = preprocess_segments(idx_train)
X_val = preprocess_segments(idx_val)
X_test = preprocess_segments(idx_test)

y_train = labels_all[idx_train]
y_val = labels_all[idx_val]
y_test = labels_all[idx_test]

src_train = sources_all[idx_train]
src_val = sources_all[idx_val]
src_test = sources_all[idx_test]

id_train = record_ids_all[idx_train]
id_val = record_ids_all[idx_val]
id_test = record_ids_all[idx_test]

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

## Datasets & DataLoaders

In [ ]:
class IMUDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            x = x * (0.80 + 0.40 * torch.rand(1))
            x = x + 0.05 * torch.randn_like(x)
            shift = torch.randint(0, x.shape[-1], (1,)).item()
            x = torch.roll(x, shifts=shift, dims=-1)
        return x, self.y[idx]


train_loader = DataLoader(
    IMUDataset(X_train, y_train, augment=True),
    batch_size=BATCH,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    IMUDataset(X_val, y_val, augment=False),
    batch_size=BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    IMUDataset(X_test, y_test, augment=False),
    batch_size=BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

train_class_counts = np.bincount(y_train, minlength=len(label_names))
class_weights = len(y_train) / (len(label_names) * train_class_counts)
class_weights = np.sqrt(class_weights)  # dampen extreme weights
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

print('Train class counts:')
for name, count in zip(label_names, train_class_counts):
    print(f'  {name:<20} {count:>4}')

## Model - SetWiseV5

`Input -> ConvStem -> Self-Attention -> Dilated TCN x2 -> GlobalAvgPool -> Linear head`

In [ ]:
class LayerNorm1d(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        return self.norm(x.transpose(1, 2)).transpose(1, 2)


class ConvStem(nn.Module):
    def __init__(self, in_ch=6, base_ch=16, depth_mult=2, temporal_kernel=31, point_kernel=9, dropout=0.1):
        super().__init__()
        hidden_ch = base_ch * depth_mult
        self.temporal = nn.Sequential(
            nn.Conv1d(in_ch, base_ch, temporal_kernel, padding=temporal_kernel // 2, bias=False),
            LayerNorm1d(base_ch),
            nn.ELU(),
        )
        self.depthwise = nn.Sequential(
            nn.Conv1d(base_ch, hidden_ch, kernel_size=1, groups=base_ch, bias=False),
            LayerNorm1d(hidden_ch),
            nn.ELU(),
            nn.Dropout(dropout),
        )
        self.pointwise = nn.Sequential(
            nn.Conv1d(hidden_ch, hidden_ch, point_kernel, padding=point_kernel // 2, bias=False),
            LayerNorm1d(hidden_ch),
            nn.ELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = self.temporal(x)
        x = self.depthwise(x)
        return self.pointwise(x)


class MHABlock(nn.Module):
    def __init__(self, d_model, num_heads=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm(x)
        h, _ = self.attn(h, h, h)
        return x + self.drop(h)


class DilatedResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5, dilation=1, dropout=0.25):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.pad1 = nn.ConstantPad1d((pad, 0), 0)
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.pad2 = nn.ConstantPad1d((pad, 0), 0)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.skip = nn.Conv1d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.drop = nn.Dropout(dropout)
        self.act = nn.ELU()

    def forward(self, x):
        h = self.drop(self.act(self.bn1(self.conv1(self.pad1(x)))))
        h = self.drop(self.act(self.bn2(self.conv2(self.pad2(h)))))
        return self.act(h + self.skip(x))


class SetWiseV5(nn.Module):
    def __init__(
        self,
        in_channels=N_CHANNELS,
        num_classes=10,
        base_ch=16,
        depth_mult=2,
        num_heads=4,
        tcn_ch=64,
        stem_dropout=0.1,
        tcn_dropout=0.25,
        head_dropout=0.35,
    ):
        super().__init__()
        hidden_ch = base_ch * depth_mult
        self.stem = ConvStem(
            in_ch=in_channels,
            base_ch=base_ch,
            depth_mult=depth_mult,
            temporal_kernel=31,
            point_kernel=9,
            dropout=stem_dropout,
        )
        self.attn = MHABlock(hidden_ch, num_heads=num_heads, dropout=stem_dropout)
        self.tcn1 = DilatedResBlock(hidden_ch, tcn_ch, kernel_size=5, dilation=1, dropout=tcn_dropout)
        self.tcn2 = DilatedResBlock(tcn_ch, tcn_ch, kernel_size=5, dilation=2, dropout=tcn_dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(tcn_ch, 64),
            nn.ReLU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        h = self.stem(x)
        h = self.attn(h.transpose(1, 2)).transpose(1, 2)
        h = self.tcn1(h)
        h = self.tcn2(h)
        h = self.pool(h).squeeze(-1)
        return self.head(h)


model = SetWiseV5(num_classes=len(label_names)).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {num_params:,}')

## Training Utilities

In [ ]:
import torch.nn.functional as F


class FocalLoss(nn.Module):
    """Focal Loss — down-weights easy examples, punishes confident mistakes.
    gamma=0 is equivalent to CrossEntropyLoss.
    gamma=2 (default) strongly penalises overconfident wrong predictions."""

    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.register_buffer('weight', weight)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none',
        )
        pt = torch.exp(-ce)  # probability of correct class
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


loss_fn = FocalLoss(weight=class_weights, gamma=2.0, label_smoothing=0.1)


def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    y_true, y_pred = [], []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        total_loss += loss.item() * xb.size(0)
        y_true.append(yb.detach().cpu().numpy())
        y_pred.append(logits.argmax(dim=1).detach().cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    acc = (y_true == y_pred).mean()
    loss = total_loss / len(loader.dataset)
    return loss, acc, y_true, y_pred


## Train Classifier

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=4,
    min_lr=LR / 50,
)

best_val_acc = -1.0
best_val_loss = float('inf')
best_epoch = 0
best_state = None
wait = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, optimizer=optimizer)
    val_loss, val_acc, _, _ = run_epoch(val_loader)
    scheduler.step(val_acc)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'lr': optimizer.param_groups[0]['lr'],
    })

    improved = (val_acc > best_val_acc) or (np.isclose(val_acc, best_val_acc) and val_loss < best_val_loss)
    if improved:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f'Epoch {epoch:>3} | '
            f'train loss {train_loss:.4f} acc {train_acc:.3f} | '
            f'val loss {val_loss:.4f} acc {val_acc:.3f} | '
            f'lr {optimizer.param_groups[0]["lr"]:.2e}'
        )

    if wait >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (best epoch: {best_epoch}).')
        break

if best_state is None:
    raise RuntimeError('Training did not produce a valid checkpoint.')

model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
print(f'Best validation accuracy: {best_val_acc:.3f} at epoch {best_epoch}')
display(history_df.tail(10))

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train', linewidth=2)
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Val', linewidth=2)
axes[0].axvline(best_epoch, color='gray', linestyle='--', alpha=0.6, label=f'Best epoch ({best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history_df['epoch'], history_df['train_acc'], label='Train', linewidth=2)
axes[1].plot(history_df['epoch'], history_df['val_acc'], label='Val', linewidth=2)
axes[1].axvline(best_epoch, color='gray', linestyle='--', alpha=0.6, label=f'Best epoch ({best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training vs Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle(f'SetWise v5 — Best val acc: {best_val_acc:.3f} @ epoch {best_epoch}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Evaluation

In [ ]:
test_loss, test_acc, y_true, y_pred = run_epoch(test_loader)

print('Test metrics:')
print(f'  loss      : {test_loss:.4f}')
print(f'  accuracy  : {test_acc:.3f}')
print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=label_names, digits=3, zero_division=0))

print('Test accuracy by source:')
for source_name in sorted(set(src_test)):
    mask = src_test == source_name
    src_acc = (y_true[mask] == y_pred[mask]).mean()
    print(f'  {source_name:<8} n={mask.sum():>4}  acc={src_acc:.3f}')

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
print('\nConfusion matrix:')
display(cm_df)

print('Test set class counts:')
for name, count in sorted(Counter(label_names[i] for i in y_true).items(), key=lambda x: (-x[1], x[0])):
    print(f'  {name:<20} {count:>4}')

## Visual Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 10))

# Normalize confusion matrix by row (true label) to show recall per class
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Recall (row-normalized)', shrink=0.8)

# Annotate cells with count and percentage
for i in range(len(label_names)):
    for j in range(len(label_names)):
        count = cm[i, j]
        pct = cm_norm[i, j]
        color = 'white' if pct > 0.5 else 'black'
        if count > 0:
            ax.text(j, i, f'{count}\n{pct:.0%}', ha='center', va='center',
                    fontsize=8, color=color, fontweight='bold' if i == j else 'normal')

ax.set_xticks(range(len(label_names)))
ax.set_yticks(range(len(label_names)))
ax.set_xticklabels(label_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(label_names, fontsize=9)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix — Test Accuracy: {test_acc:.1%}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Per-class accuracy bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)
sort_idx = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if a < 0.4 else '#f39c12' if a < 0.6 else '#2ecc71' for a in per_class_acc[sort_idx]]
bars = ax.barh(range(len(label_names)), per_class_acc[sort_idx], color=colors)
ax.set_yticks(range(len(label_names)))
ax.set_yticklabels(label_names[sort_idx], fontsize=10)
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Per-Class Test Accuracy', fontsize=13, fontweight='bold')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlim(0, 1.05)

for bar, acc in zip(bars, per_class_acc[sort_idx]):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{acc:.0%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()


## Example Inference

In [ ]:
model.eval()
rows = []
for i in range(min(15, len(X_test))):
    xb = torch.tensor(X_test[i:i+1].transpose(0, 2, 1), dtype=torch.float32, device=device)
    with torch.no_grad():
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
    pred_idx = int(probs.argmax(dim=1).item())
    rows.append({
        'record_id': id_test[i],
        'source': src_test[i],
        'true_exercise': label_names[y_test[i]],
        'pred_exercise': label_names[pred_idx],
        'confidence': float(probs.max().item()),
    })

display(pd.DataFrame(rows))